# 1. Import

In [13]:
import pandas as pd
import numpy as np

# 2. Load Data

In [14]:
# หมายเหตุสำคัญ: ค่าที่หายไปในไฟล์นี้ถูกเก็บเป็น string '?' เท่านั้น
# แต่คอลัมน์ max_glu_serum / A1Cresult มีค่า "None" ที่เป็นหมวดหมู่จริง
# (แปลว่า "ไม่ได้ตรวจ") ถ้าปล่อยให้ pandas ใช้ default NaN parsing มันจะ
# แปลง "None" เป็น NaN ไปด้วย (ผิดความหมาย) จึงต้องปิด default na parsing
# แล้วระบุเองว่ามีแค่ '?' เท่านั้นที่ถือเป็นค่าหายไป
df = pd.read_csv("../data/diabetic_data.csv", keep_default_na=False, na_values=["?"])

df.shape

C:\Users\HP\AppData\Local\Temp\ipykernel_23348\1776909274.py:6: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/diabetic_data.csv", keep_default_na=False, na_values=["?"])


(101766, 50)

# 3. Remove Invalid Rows

In [15]:
# gender มีค่า 'Unknown/Invalid' อยู่ 3 แถว -> ตัดทิ้ง
df = df[df["gender"] != "Unknown/Invalid"]

In [16]:
# discharge_disposition_id ที่หมายถึง "เสียชีวิต" หรือ "hospice" (11, 13, 14,
# 19, 20, 21) ทำให้ไม่มีทางถูก readmit ได้จริง เก็บไว้จะรบกวนโมเดล -> ตัดทิ้ง
death_hospice_codes = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(death_hospice_codes)]
print(f"Shape after removing death/hospice discharges: {df.shape}")

Shape after removing death/hospice discharges: (99340, 50)


# 4. Patient encounters (แก้ไข)

**เดิม** เราตัดผู้ป่วยที่มาซ้ำออกด้วย `drop_duplicates(subset="patient_nbr", keep="first")` เพื่อกัน data leakage — แต่วิธีนี้ทิ้งข้อมูลไปถึง ~31% (101,766 → 69,970 แถว) และทิ้ง signal ที่มีประโยชน์ที่สุดออกไปด้วย เพราะผู้ป่วยที่กลับมาหลายครั้งคือกลุ่มที่มีแนวโน้ม readmit สูงอยู่แล้ว

**ใหม่**: เก็บทุก encounter ไว้ทั้งหมด แต่**คง `patient_nbr` ไว้ในไฟล์**เพื่อให้ขั้นตอน train/test split (ในไฟล์ `02_model_preparation`) ใช้ `GroupShuffleSplit` แบ่งตาม `patient_nbr` แทนการแบ่งแบบสุ่มรายแถว — วิธีนี้ได้ข้อมูลครบ ไม่ทิ้ง signal และยังไม่มี leakage (ผู้ป่วยคนเดียวกันจะไม่ไปโผล่ทั้ง train และ test)

⚠️ **สำคัญ**: ห้ามลบ `patient_nbr` ออกในไฟล์นี้ ให้ตัดออกทีหลังในไฟล์ model_preparation หลังจากใช้แบ่ง group เสร็จแล้วเท่านั้น

In [17]:
print(f"Shape (all encounters kept): {df.shape}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")

Shape (all encounters kept): (99340, 50)
Unique patients: 69987


# 5. Feature Engineering (เพิ่มใหม่)

In [18]:
# จำนวนครั้งที่มาโรงพยาบาลก่อนหน้า (outpatient/emergency/inpatient) เป็นหนึ่งใน
# ตัวทำนายการ readmit ที่แข็งแรงที่สุดในวรรณกรรมทางการแพทย์ แต่เดิม pipeline
# ไม่ได้ใช้ประโยชน์จาก 3 คอลัมน์นี้เลยนอกจากเป็นฟีเจอร์ดิบ จึงเพิ่ม feature สรุปไว้
df["total_prior_visits"] = (
    df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
)
df["had_prior_inpatient"] = (df["number_inpatient"] > 0).astype(int)

# 6. Drop / Reduce Columns

In [19]:
# - weight: missing ~97% ของข้อมูล ใช้ประโยชน์แทบไม่ได้ -> ตัดทิ้ง
# - payer_code: missing ~40% และไม่เกี่ยวข้องทางการแพทย์กับ readmission -> ตัดทิ้ง
# - encounter_id: เป็น ID ไม่ใช่ feature เชิงทำนาย -> ตัดทิ้ง
# - patient_nbr: **ไม่ตัดในไฟล์นี้แล้ว** เก็บไว้ให้ 02_model_preparation ใช้ทำ
#   GroupShuffleSplit ก่อน แล้วค่อยตัดออกจาก X ทีหลัง
# - medical_specialty: เดิมตัดทิ้งเพราะ missing ~49% แต่ 51% ที่เหลือมีข้อมูล
#   ที่มีประโยชน์ (แผนกที่รักษาสัมพันธ์กับความเสี่ยง readmit) จึง**เก็บไว้แทน**
#   โดยลด cardinality ด้วยการรวมแผนกที่พบน้อยเป็น 'Other' และเติม missing เป็น
#   'Missing' (เป็นหมวดหมู่ที่มีความหมายในตัวเอง ไม่ใช่แค่ค่าว่าง)
drop_cols = ["weight", "payer_code", "encounter_id"]
df.drop(columns=drop_cols, inplace=True)

df["medical_specialty"] = df["medical_specialty"].fillna("Missing")
top_specialties = df["medical_specialty"].value_counts().nlargest(10).index
df["medical_specialty"] = df["medical_specialty"].where(
    df["medical_specialty"].isin(top_specialties), "Other"
)
print(df["medical_specialty"].value_counts())

medical_specialty
Missing                       48614
InternalMedicine              14237
Other                          8199
Emergency/Trauma               7419
Family/GeneralPractice         7252
Cardiology                     5278
Surgery-General                3059
Nephrology                     1539
Orthopedics                    1392
Orthopedics-Reconstructive     1230
Radiologist                    1121
Name: count, dtype: int64


# 7. Diagnosis Grouping

In [20]:
def map_diagnosis(code):
    if pd.isna(code):
        return "Missing"
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    try:
        val = float(code)
    except ValueError:
        return "Other"
    if 390 <= val <= 459 or val == 785:
        return "Circulatory"
    elif 460 <= val <= 519 or val == 786:
        return "Respiratory"
    elif 520 <= val <= 579 or val == 787:
        return "Digestive"
    elif 250 <= val < 251:
        return "Diabetes"
    elif 800 <= val <= 999:
        return "Injury"
    elif 710 <= val <= 739:
        return "Musculoskeletal"
    elif 580 <= val <= 629 or val == 788:
        return "Genitourinary"
    elif 140 <= val <= 239:
        return "Neoplasms"
    else:
        return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].apply(map_diagnosis)

# race ที่เหลือมี missing เล็กน้อย (~2%) -> เติมเป็นหมวด 'Missing'
# (ลบ cell ที่ fillna ซ้ำสองรอบของไฟล์เดิมออก เหลือรอบเดียวพอ)
df["race"] = df["race"].fillna("Missing")

# 8. Drop Constant Columns

In [21]:
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print(f"Dropping constant columns: {constant_cols}")
df.drop(columns=constant_cols, inplace=True)

Dropping constant columns: ['examide', 'citoglipton']


# 9. Target Variable

In [22]:
# readmitted มี 3 ค่า: 'NO', '<30', '>30'
# ตั้งเป็นปัญหา binary classification: ทำนายว่าผู้ป่วยจะ readmit ภายใน 30 วันหรือไม่
#   1 = readmitted ภายใน 30 วัน ('<30')
#   0 = ไม่ readmit หรือ readmit หลัง 30 วัน ('NO', '>30')
df["readmitted"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)

print("\nTarget distribution:")
print(df["readmitted"].value_counts(normalize=True))


Target distribution:
readmitted
0    0.886108
1    0.113892
Name: proportion, dtype: float64


# 10. Final Check & Save

In [23]:
remaining_na = df.isna().sum()
print("Remaining missing values per column (should be empty):")
print(remaining_na[remaining_na > 0])

Remaining missing values per column (should be empty):
Series([], dtype: int64)


In [24]:
df.to_csv("../data/diabetic_data_cleanedv2.csv", index=False)
df.shape

(99340, 47)